# Example sheet for robot arm tasks

This work sheet will take you through how the robot gym environment works, and how you can generate a task, get information about the environment and act on it. 

In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import sys
sys.path.append("../..")
from Simulation.PyBullet.environment import *
from Simulation.PyBullet.tasks import *
from Controller.languageModel import *

## Booting the simulation 
Do this once to boot the simulator. Once it is booted you do not need to call Env again. If you want to put it back to original settings, you can call env.reset()

In [2]:
env=Env(realtime=1,speed=8)
#boots up the environment 
env.reset()

### Moving the arm
The arm uses inverse kinematics to move to specific points in the environment. Have a go at changing the coordinates to move the arm around. The units of the coordinates are in meters.

What are the constraints? What happens if you go out of the bounds? Understanding the limitations of robotics is important.

In [3]:
target_coord=[0.4,-0.3,0.2] #change the values
env.move_gripper_to(target_coord)
env.step(100) #update sim

Lets add an object to the environment,we can pick up an object by hovering over it with ```env.move_gripper_to``` and calling a function ```pick_block```

In [4]:
env.reset()
#generate the block in the simulation
env.generate_blocks(1) #generate 1 block in a random place
cube_pos, _ = p.getBasePositionAndOrientation(env.block_ids[0]) #get information about this block
cube_pos=list(cube_pos)
cube_pos[2]+=0.08 #increase the z position so the robot hovers over it and does not crush it
env.move_gripper_to(cube_pos)

env.pick_block(env.block_ids[0]) #pick it up
up=np.array(cube_pos)
up[2]+=0.6 #increase height to show it is attached
env.move_gripper_to(up)
env.step(100) #update sim

You can drop the block by calling ```env.put_block() ```

In [5]:
env.put_block()
env.step(100) #update sim

Finally we can get details about the environment by calling the observation function. It returns a dictionary of items

In [6]:
dictionary=env.get_observation()
print("Block coordinates:",dictionary['blocks'])
print("Block colours:",dictionary['block_colours'])
print("Block sizes:",dictionary['sizes'])

print("All data items",list(dictionary.keys()))

Block coordinates: [(0.5001556420835115, 2.5718474500556497e-06, 0.023754118231847754)]
Block colours: [(0.5671306516623507, 0.3247738750760214, 0.607158150622296, 1.0)]
Block sizes: [1]
All data items ['blocks', 'block_colours', 'robot_end_position', 'holding_constraint', 'block_name', 'sizes', 'contacts', 'holding']


Play around with the functions, try generate more blocks and select different ones. Here are a few challenges to help you get used to the robot control

In [7]:
# Make the robot pick up block and move it to a different location

In [8]:
# Make the robot generate two blocks and place one on top of the other

## Hard code challenges
Here are a few preset challenges. Can you hard code a solution that works? You can make use of the get observation function to look at objects and information such as colours and sizes, then use the coordinates to move to each. 

### Task 1 - Arrange into a tower
![My GIF](https://raw.githubusercontent.com/shepai/Robot_shape_learning/refs/heads/main/Assets/Gifs/task1_fast.gif)

In [9]:
#Task 1: arrange into a tower


### Task 2 - sort the colours into three seperate towers

![My GIF](https://raw.githubusercontent.com/shepai/Robot_shape_learning/refs/heads/main/Assets/Gifs/task2_fast.gif)

In [10]:
# Task 2

## Using AI
We could use reinforcment learning or genetic algorithms to optimize neural controllers. This would take a large amount of time. Can we make use of existing large language models for robotic control?

You will need to make sure to install the ollama library from 
https://ollama.com/download/windows

or for linux
```[bash]
pip install ollama
curl -fsSL https://ollama.com/install.sh | sh
ollama pull gemma3
ollama pull llama3
```
You can chose either model as long as you use the one downloaded

```[bash]
ollama pull gemma3
ollama pull llama3
```
Have a look at the prompt we are using for control

In [11]:
decision = Decisions(model="gemma3")

the_prompts = decision.system_prompt
print(the_prompts)



['You are a robot arm which can pick up objects with its tractor beam on the hand. You just need to hover the gripper over the z with an offset of 0.15 where the coordinates are in the form (x,y,z) otherwise you crush the object', 'Your only actions you can do are move_gripper_to(target_coord) to move the gripper hand to a specific coordinate, pick_block(index) to pick up a specific block, put_block() to drop whatever is held. Given a task generate the steps to solving it using only these listed commands', ' ']


We will also need to be able to convert our observations of the environment into a text prompt. We have written the code for this for you. Feel free to play around if you can think of a better way to do this?

In [13]:
def observations_to_text(obs):
    string="Environment information"
    for key in obs:
        if type(obs[key])==type([]):
            string+=key+": "
            for i in range(len(obs[key])):
                string+=str(obs[key][i])+", "
        else:
            string+=key+": "+str(obs[key])
        string+="\n"
    return string
print(observations_to_text(env.get_observation()))

Environment informationblocks: (0.5001556420835115, 2.5718474500556497e-06, 0.023754118231847754), 
block_colours: (0.5671306516623507, 0.3247738750760214, 0.607158150622296, 1.0), 
robot_end_position: (0.5000313875544178, -5.123351179911375e-06, 0.7099992578096335)
holding_constraint: None
block_name: cube_small.urdf, 
sizes: 1, 
contacts: [], 
holding: None



The prompt is automattically used, but you can change it by changing the_prompts [0] and [1] like so
```[python]
the_prompt[0]="new prompt"
the prompt[1]="new prompt"
```

We set the task and generate action using the following:

In [ ]:
env.reset()
env.generate_blocks(2) #generate 2 block in a random place
decision.set_task("You are given two blocks and you need to stack them on top of eachother")
reply=decision.chat(observations_to_text(env.get_observation()))
print(reply)

Okay, I understand. I have the following information:

*   **Blocks:** I have a block with coordinates (0.5001556420835115, 2.5718474500556497e-06, 0.023754118231847754).
*   **Block Colors:** The block is red (approximately).
*   **Robot End Position:** The robot's end position is (0.5000313875544178, -5.123351179911375e-06, 0.7099992578096335).
*   **Holding Constraint:** None.
*   **Block Name:** `cube_small.urdf`
*   **Size:** 1
*   **Contacts:** None.
*   **Holding:** None

I will now proceed to pick up the block and place it on top of the other block.

```
pick_block(0)
move_gripper_to(0.5000313875544178, -5.123351179911375e-06, 0.7099992578096335)
put_block()
```


### Task - write code that locates the functions it is outputting in order, and get the robot to run through them

In [ ]:
# hint use a library called re